In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_13.pickle

In [ ]:
%%cudf.pandas.profile
### cell 13 ###
def grab_subset_of_data_25(df, question_of_interest):
    # Select only the columns whose names contain the question (GPU-side via boolean mask)
    mask = df.columns.str.contains(question_of_interest)
    sub_df = df.loc[:, mask]

    # Remove everything up to the last dash and both unwanted substrings in one GPU call
    pattern = r".*-\s*|\s*\(direct from AWS, Azure, GCP, or similar\)|\s*\(resulting in a university degree\)"
    sub_df.columns = sub_df.columns.str.replace(pattern, "", regex=True)

    # Drop rows where all values in these columns are null using GPU operations
    non_null_rows = sub_df.notna().any(axis=1)
    return sub_df.loc[non_null_rows]

question_of_interest_cell25 = 'On which platforms have you begun or completed data science courses?'
online_learning_platforms_df_2022 = grab_subset_of_data_25(
    responses_df_2022_cell10,
    question_of_interest_cell25
)

online_learning_platforms_df_2022.info()